In [6]:
# CELL 1 — PARAMETERS  ← tag this cell
batch_id = "MANUAL_RUN"

StatementMeta(, 934f63f3-89ed-4a13-a2b3-11572928d447, 8, Finished, Available, Finished, False)

In [7]:
# CELL 2
from datetime import datetime, timezone
from pyspark.sql import functions as F

run_ts = datetime.now(timezone.utc)
refs = {
    "icd10ca": "ref_icd10ca",
    "loinc": "ref_loinc",
    "medication": "ref_medication",
    "payer": "ref_payer",
    "code_mapping": "ref_code_mapping",
}

for src, tgt in refs.items():
    df = (spark.read.option("header", True).option("inferSchema", False)
          .csv(f"Files/landing/reference/{src}.csv"))
    df = (df.withColumn("_source_system", F.lit("REF"))
            .withColumn("_batch_id", F.lit(batch_id))
            .withColumn("_ingest_ts", F.lit(run_ts))
            .withColumn("_load_date", F.lit(run_ts.date())))
    # Reference data is overwrite, not append. There is one current version
    # of ICD-10-CA, not a history of them. This is the only place in Bronze
    # where overwrite is correct.
    df.write.format("delta").mode("overwrite") \
      .option("overwriteSchema", "true").saveAsTable(tgt)
    print(f"{tgt:<24} {df.count():>7,} rows")

StatementMeta(, 934f63f3-89ed-4a13-a2b3-11572928d447, 9, Finished, Available, Finished, False)

ref_icd10ca                   32 rows
ref_loinc                     25 rows
ref_medication                28 rows
ref_payer                      8 rows
ref_code_mapping              31 rows


In [1]:
total = 0
for t in sorted(x.tableName for x in spark.sql("SHOW TABLES").collect()):
    if t.startswith("stg_"):
        print(f"LEFTOVER STAGING: {t}")
        continue
    n = spark.table(t).count()
    total += n
    print(f"{t:<40} {n:>10,}")
print(f"\nTotal: {total:,}")

StatementMeta(, dc430d01-95d9-4437-88b9-a193a6919fd3, 3, Finished, Available, Finished, False)

claims_835_remittance                         1,422
claims_837_service_line                       6,468
ehr_admission                                11,568
ehr_bed_assignment                           14,804
ehr_diagnosis                                37,318
ehr_emergency_visit                          25,037
ehr_patient                                  25,998
facil_bed                                       101
facil_department                                 49
facil_hospital                                    5
fin_invoice                                  11,568
fin_invoice_line                             52,196
fin_patient_account                          15,985
hr_doctor                                       240
lis_lab_result                              323,456
pharm_medication_order                       97,511
ref_code_mapping                                 31
ref_icd10ca                                      32
ref_loinc                                        25
ref_medicati